# LeetCode 4013 — Count Subarrays With Even Odd Ratio II bằng Python

- Link: https://leetcode.com/problems/count-subarrays-with-even-odd-ratio-ii/
- Độ khó: **Hard**
- Chủ đề: Tổng tiền tố (prefix sum), Biến đổi bất đẳng thức tỉ lệ, Đếm cặp thuận thế (BIT / merge sort), Nén toạ độ

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countRatioSubarrays(self, nums: list[int], a: int, b: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Với mỗi đoạn con, gọi `x` là số phần tử **chẵn** và `y` là số phần tử **lẻ**. Đoạn con **hợp lệ** nếu `y > 0` và `x / y ≤ a / b` (so sánh bằng giá trị hữu tỉ **chính xác**). Đếm số đoạn con hợp lệ.

Bản `n ≤ 10^5` không cho phép duyệt mọi đoạn con. Điểm hay của bài: bất đẳng thức tỉ lệ `x / y ≤ a / b` viết lại thành **một tổng tuyến tính** `a·y − b·x ≥ 0`, và điều kiện `y > 0` **tự động thoả** khi bất đẳng thức đó đúng (đoạn toàn số chẵn cho `a·y − b·x < 0`). Biến mỗi số lẻ thành `+a`, mỗi số chẵn thành `−b`, bài trở thành: **đếm số đoạn con có tổng ≥ 0** — tức đếm số cặp tiền tố `L < R` với `s[L] ≤ s[R]`.

Sau notebook này bạn sẽ:

1. So sánh hai phân số bằng **nhân chéo** (không dùng số thực).
2. Biến điều kiện tỉ lệ + `y > 0` thành một điều kiện tuyến tính duy nhất, và hiểu vì sao hợp nhất được.
3. Dùng **tổng tiền tố** để đưa "tổng đoạn ≥ 0" về "so sánh hai tiền tố".
4. Đếm cặp `L < R` có `s[L] ≤ s[R]` bằng **Fenwick tree + nén toạ độ** và bằng **sắp xếp trộn (merge sort)**.
5. Nhận ra bẫy khi đổi chiều bất đẳng thức (`≥`): điều kiện `y > 0` **không** còn tự động đúng.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `nums` và hai số nguyên dương `a`, `b`. Với một đoạn con, `x` = số phần tử chẵn, `y` = số phần tử lẻ. Đoạn con **hợp lệ** nếu:

1. `y > 0`, và
2. `x / y ≤ a / b` (so sánh chính xác các giá trị hữu tỉ).

Trả về số đoạn con (liên tiếp, không rỗng) hợp lệ.

**Ví dụ 1:** `nums = [1,2,1,2]`, `a = 3`, `b = 2` → `7`. Các đoạn hợp lệ: `[1]`, `[1,2]`, `[1,2,1]`, `[1,2,1,2]`, `[2,1]`, `[1]` (vị trí 2), `[1,2]` (vị trí 2..3).

**Ví dụ 2:** `nums = [2,2,1]`, `a = 2`, `b = 1` → `3`.

**Ví dụ 3:** `nums = [2,2,2]`, `a = 1`, `b = 1` → `0` (mọi đoạn con không có số lẻ).

**Ràng buộc**

- `1 <= nums.length <= 10^5`
- `1 <= nums[i] <= 10^9`, `1 <= a, b <= 10^9`

In [1]:
from fractions import Fraction
from bisect import bisect_left
import random, time

EXAMPLES = [
    ([1, 2, 1, 2], 3, 2, 7),
    ([2, 2, 1], 2, 1, 3),
    ([2, 2, 2], 1, 1, 0),
]
for nums, a, b, e in EXAMPLES:
    print(f"nums={nums}, a={a}, b={b} -> kỳ vọng {e}")

nums=[1, 2, 1, 2], a=3, b=2 -> kỳ vọng 7
nums=[2, 2, 1], a=2, b=1 -> kỳ vọng 3
nums=[2, 2, 2], a=1, b=1 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Chẵn / lẻ: `x % 2`

`x % 2 == 0` ⇔ `x` chẵn; `x % 2 == 1` ⇔ `x` lẻ. Với số dương, `x % 2` chính là 0 hoặc 1.

In [2]:
nums = [1, 2, 7, 10, 3]
print([x % 2 for x in nums])                           # 1 = lẻ, 0 = chẵn
evens = sum(1 for x in nums if x % 2 == 0)
odds = len(nums) - evens
print("chẵn:", evens, "lẻ:", odds)

[1, 0, 1, 0, 1]
chẵn: 2 lẻ: 3


### 3.2. So sánh phân số chính xác bằng nhân chéo

Với mẫu số dương: `x / y ≤ a / b` ⇔ `x · b ≤ a · y`. Python có số nguyên lớn nên phép nhân không tràn; `fractions.Fraction` cũng so sánh chính xác (nhưng chậm hơn). Tuyệt đối tránh số thực (`float`) vì sai số làm sai kết quả khi tử/mẫu lớn.

In [3]:
x, y, a, b = 2, 2, 3, 2
print(x / y <= a / b, "|", x * b <= a * y, "|", Fraction(x, y) <= Fraction(a, b))
# float có thể sai với số lớn:
big_a, big_b = 10**18, 10**18 + 1                # big_a / big_b < 1 một chút
print(1 / 1 <= big_a / big_b, Fraction(1, 1) <= Fraction(big_a, big_b))   # float làm tròn big_a/big_b thành 1.0 -> True (sai); chính xác: False

True | True | True
True False


### 3.3. Biến đổi `x / y ≤ a / b` thành `a·y − b·x ≥ 0`

Vì `y > 0` và `b > 0`: `x / y ≤ a / b` ⇔ `x·b ≤ a·y` ⇔ `a·y − b·x ≥ 0`. Vế phải là **tuyến tính** theo `x, y`: nếu mỗi số lẻ đóng góp `+a` và mỗi số chẵn đóng góp `−b` thì `a·y − b·x` chính là **tổng của đoạn con** sau khi thay số.

**Trường hợp `y = 0`:** đoạn không rỗng nên `x ≥ 1`, và `a·0 − b·x = −b·x < 0`: bất đẳng thức **không đúng**. Vì vậy điều kiện "`y > 0`" đã nằm trong "`a·y − b·x ≥ 0`" — ta chỉ cần một điều kiện.

In [4]:
def transform(nums, a, b):
    return [a if x % 2 else -b for x in nums]

nums, a, b = [1, 2, 1, 2], 3, 2
arr = transform(nums, a, b)
print(arr)                              # [3, -2, 3, -2]
# đoạn [1, 2, 1] có x=1, y=2: a*y - b*x = 3*2 - 2*1 = 4 = tổng 3 - 2 + 3
print(sum(arr[0:3]), 3 * 2 - 2 * 1)
# đoạn toàn chẵn [2, 2]: x=2, y=0 -> -b*x < 0
print(sum(arr[1:2]), "(đoạn [2] toàn chẵn: tổng âm)")

[3, -2, 3, -2]
4 4
-2 (đoạn [2] toàn chẵn: tổng âm)


### 3.4. Tổng tiền tố: tổng đoạn = hiệu hai tiền tố

`s[0] = 0`, `s[i+1] = s[i] + arr[i]`. Tổng của `arr[L..R−1]` là `s[R] − s[L]` (với `0 ≤ L < R ≤ n`). Điều kiện "tổng ≥ 0" ⇔ `s[L] ≤ s[R]`. Mỗi đoạn con khác rỗng ứng với **đúng một** cặp `(L, R)` với `L < R`.

In [5]:
def prefix_sums(arr):
    s = [0]
    for v in arr:
        s.append(s[-1] + v)
    return s

arr = [3, -2, 3, -2]
s = prefix_sums(arr)
print(s)                                    # [0, 3, 1, 4, 2]
L, R = 1, 3
print(sum(arr[L:R]), s[R] - s[L])           # cùng 1

[0, 3, 1, 4, 2]
1 1


### 3.5. Nén toạ độ (coordinate compression)

Giá trị `s[i]` có thể rất lớn (tới `10^14`) nhưng chỉ có `n + 1` giá trị. Ta gán mỗi giá trị một **hạng** nhỏ `1..m` giữ nguyên thứ tự: `ranks = sorted(set(s))`, hạng của `v` là `bisect_left(ranks, v) + 1`. Nhờ đó dùng được mảng/BIT cỡ `m`.

In [6]:
s = [0, 3, 1, 4, 2, 3, 1]
ranks = sorted(set(s))
print(ranks)
print([bisect_left(ranks, v) + 1 for v in s])     # hạng 1-based của từng giá trị
# so sánh hạng khớp so sánh giá trị:
vals = [10**14, -10**14, 5]
rk = sorted(set(vals))
print([bisect_left(rk, v) + 1 for v in vals])

[0, 1, 2, 3, 4]
[1, 4, 2, 5, 3, 4, 2]
[3, 1, 2]


### 3.6. Cây Fenwick (BIT) và merge sort đếm cặp

- **BIT:** `update(i, +1)` đánh dấu một giá trị đã gặp; `query(i)` = số giá trị đã gặp có hạng `≤ i` — mỗi thao tác `O(log n)`.
- **Merge sort:** khi trộn hai nửa đã sắp, ta đếm cặp "trái ≤ phải" bằng hai con trỏ — gọi là **đếm cặp thuận thế** (ngược với đếm nghịch thế).

In [7]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)
    def update(self, i, delta):          # i: 1-based
        while i <= self.n:
            self.t[i] += delta
            i += i & -i
    def query(self, i):                  # tổng a[1..i]
        s = 0
        while i > 0:
            s += self.t[i]
            i -= i & -i
        return s

bit = Fenwick(5)
for rank in [3, 1, 3, 5]:
    print(f"gặp hạng {rank}: đã có {bit.query(rank)} giá trị <= nó", end=" | ")
    bit.update(rank, 1)
print()

gặp hạng 3: đã có 0 giá trị <= nó | gặp hạng 1: đã có 0 giá trị <= nó | gặp hạng 3: đã có 2 giá trị <= nó | gặp hạng 5: đã có 3 giá trị <= nó | 


## 4. Ý tưởng nền tảng / trực giác

**Tính tay ví dụ 1:** `nums = [1,2,1,2]`, `a = 3`, `b = 2`.

- `arr = [3, −2, 3, −2]` (lẻ → `+3`, chẵn → `−2`).
- `s = [0, 3, 1, 4, 2]`.
- Đếm cặp `(L, R)`, `L < R`, `s[L] ≤ s[R]`:

| `R` | `s[R]` | các `L < R` có `s[L] ≤ s[R]` | số cặp |
|---|---|---|---|
| 1 | 3 | `L = 0` (0) | 1 |
| 2 | 1 | `L = 0` (0); `L = 1` (3 > 1 ✗) | 1 |
| 3 | 4 | `L = 0, 1, 2` | 3 |
| 4 | 2 | `L = 0` (0); `L = 1` (3 ✗); `L = 2` (1); `L = 3` (4 ✗) | 2 |

Tổng `1 + 1 + 3 + 2 = 7`. ✔

**Ví dụ 2:** `[2,2,1]`, `a = 2`, `b = 1`: `arr = [−1, −1, 2]`, `s = [0, −1, −2, 0]`. Chỉ `R = 3` (`s = 0`) có cặp: `L = 0, 1, 2` đều `≤ 0` ⇒ 3. ✔

## 5. Cách 1 — Vét cạn: duyệt mọi đoạn con, so sánh phân số chính xác

**Ý tưởng (nguyên văn đề).** Với mỗi `(i, j)`, đếm `x` (chẵn), `y` (lẻ) bằng cộng dồn khi mở rộng `j`; hợp lệ nếu `y > 0` và `Fraction(x, y) ≤ Fraction(a, b)`.

In [8]:
def count_bruteforce(nums, a, b):
    n = len(nums)
    ratio = Fraction(a, b)
    total = 0
    for i in range(n):
        x = y = 0
        for j in range(i, n):
            if nums[j] % 2 == 0:
                x += 1
            else:
                y += 1
            if y > 0 and Fraction(x, y) <= ratio:
                total += 1
    return total

for nums, a, b, e in EXAMPLES:
    print(count_bruteforce(nums, a, b), e)

7 7
3 3
0 0


**Độ phức tạp.** `O(n²)` đoạn con, mỗi lần so sánh `Fraction` (chậm). Với `n = 10^5` là `5·10^9` — quá chậm.

**Nút thắt:** ta so sánh phân số cho từng đoạn. Nếu đưa về tổng tuyến tính, có thể dùng tiền tố.

## 6. Bước đột phá

**Insight 1 — một bất đẳng thức tuyến tính.** `y > 0 ∧ x/y ≤ a/b` ⇔ `a·y − b·x ≥ 0` (mục 3.3). Đoạn toàn chẵn có `a·y − b·x = −b·x < 0` nên bị loại tự nhiên.

**Insight 2 — tổng tiền tố.** Gán lẻ → `+a`, chẵn → `−b`; tổng đoạn `(L, R]` là `s[R] − s[L]`. Điều kiện ⇔ `s[L] ≤ s[R]`.

**Insight 3 — đếm cặp thuận thế.** Đáp án `= #{(L, R): 0 ≤ L < R ≤ n, s[L] ≤ s[R]}`. Duyệt `R` từ trái sang phải, cần biết **có bao nhiêu `s[L]` đã gặp mà `≤ s[R]`** — đúng việc của BIT trên giá trị đã nén. Hoặc dùng merge sort.

**Quy tắc nếu… thì…**

- Nếu tổng đoạn `≥ 0` ⇒ hợp lệ (tính cả trường hợp bằng đúng `0`, ứng với `x / y = a / b`).
- Nếu bằng nhau `s[L] = s[R]` ⇒ **tính** (dấu `≤`).
- Nếu đổi chiều thành `x/y ≥ a/b` thì `y > 0` **không** còn tự động đúng (xem phần 13/15).

**Bất biến BIT.** Trước khi xử lý `R`, BIT chứa đúng các giá trị `s[0..R−1]`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tổng tiền tố, vẫn `O(n²)`

Sau khi biết điều kiện `s[L] ≤ s[R]`, ta có thể đếm trực tiếp mọi cặp `(L, R)` với hai vòng lặp, không còn so sánh phân số. Nhanh hơn Cách 1 rất nhiều ở hằng số, nhưng vẫn `O(n²)`; dùng làm trung gian để kiểm chứng phép biến đổi.

In [9]:
def count_prefix_quadratic(nums, a, b):
    s = [0]
    for x in nums:
        s.append(s[-1] + (a if x % 2 else -b))
    n = len(nums)
    total = 0
    for R in range(1, n + 1):
        for L in range(R):
            if s[L] <= s[R]:
                total += 1
    return total

for nums, a, b, e in EXAMPLES:
    print(count_prefix_quadratic(nums, a, b), e)

7 7
3 3
0 0


### 7.2. Cách 3 — BIT trên tiền tố đã nén toạ độ (bản nộp)

**Giải thích từng đoạn.**

1. Tính `s[0..n]`.
2. Nén: `ranks = sorted(set(s))`; hạng của `v` là `bisect_left(ranks, v) + 1`.
3. Duyệt `v` trong `s` theo thứ tự `R = 0..n`: `ans += bit.query(rank(v))` (số `s[L]` đã gặp `≤ v`), rồi `bit.update(rank(v), 1)`.

Ở `R = 0` BIT trống nên cộng 0 — đúng vì chưa có `L < 0`.

In [10]:
def count_bit(nums, a, b):
    s = [0]
    for x in nums:
        s.append(s[-1] + (a if x % 2 else -b))
    ranks = sorted(set(s))
    bit = Fenwick(len(ranks))
    ans = 0
    for v in s:
        r = bisect_left(ranks, v) + 1
        ans += bit.query(r)          # số tiền tố đã gặp <= v (kể cả bằng)
        bit.update(r, 1)
    return ans

for nums, a, b, e in EXAMPLES:
    print(count_bit(nums, a, b), e)

7 7
3 3
0 0


**Độ phức tạp.** Sắp xếp/nén `O(n log n)`, mỗi phần tử một `query` và một `update` `O(log n)` ⇒ `O(n log n)` thời gian, `O(n)` bộ nhớ.

### 7.3. Cách 4 — Chia để trị: merge sort đếm cặp thuận thế

**Ý tưởng.** Chia `s` thành nửa trái, nửa phải; đệ quy đếm cặp trong từng nửa (và sắp chúng tăng dần). Đếm các cặp **chéo** `(L ở trái, R ở phải)` với `s[L] ≤ s[R]`: vì hai nửa đã sắp, với mỗi phần tử phải `v` ta tiến con trỏ `i` trên nửa trái đến khi `left[i] > v`; khi đó `i` chính là số phần tử trái `≤ v`. Cuối cùng trộn hai nửa. Thứ tự chỉ số trái/phải được giữ vì đệ quy chia theo vị trí.

In [11]:
def count_merge_sort(nums, a, b):
    s = [0]
    for x in nums:
        s.append(s[-1] + (a if x % 2 else -b))

    def sort_count(arr):
        n = len(arr)
        if n <= 1:
            return arr, 0
        mid = n // 2
        left, cl = sort_count(arr[:mid])
        right, cr = sort_count(arr[mid:])
        count = cl + cr
        i = 0
        for v in right:                       # right đã sắp tăng
            while i < len(left) and left[i] <= v:
                i += 1
            count += i                        # số phần tử trái <= v
        merged, p, q = [], 0, 0
        while p < len(left) and q < len(right):
            if left[p] <= right[q]:
                merged.append(left[p]); p += 1
            else:
                merged.append(right[q]); q += 1
        merged.extend(left[p:]); merged.extend(right[q:])
        return merged, count

    return sort_count(s)[1]

for nums, a, b, e in EXAMPLES:
    print(count_merge_sort(nums, a, b), e)

7 7
3 3
0 0


**Độ phức tạp.** `T(n) = 2T(n/2) + O(n)` ⇒ `O(n log n)` thời gian, `O(n)` bộ nhớ phụ. Hằng số trong Python hơi lớn hơn BIT nhưng không cần nén toạ độ.

## 8. TRACE từng bước

### 8.1. Trace BIT cho ví dụ 1

`s = [0, 3, 1, 4, 2]`; `ranks = [0, 1, 2, 3, 4]` nên hạng bằng giá trị + 1. Mỗi bước: in `v`, hạng, `query` (số đã gặp `≤ v`), tổng chạy và nội dung BIT dưới dạng "multiset đã gặp".

In [12]:
def trace_count_bit(nums, a, b):
    s = [0]
    for x in nums:
        s.append(s[-1] + (a if x % 2 else -b))
    print("arr =", [a if x % 2 else -b for x in nums], "| s =", s)
    ranks = sorted(set(s))
    print("ranks =", ranks)
    bit = Fenwick(len(ranks))
    seen = []
    ans = 0
    for R, v in enumerate(s):
        r = bisect_left(ranks, v) + 1
        c = bit.query(r)
        partners = [L for L in range(R) if s[L] <= v]
        ans += c
        print(f"R={R}: s[R]={v}, hạng={r}, đã gặp {seen}, query -> {c} (các L = {partners}), tổng = {ans}")
        assert c == len(partners)
        bit.update(r, 1)
        seen.append(v)
    print("Đáp án =", ans)
    return ans

trace_count_bit(*EXAMPLES[0][:3])

arr = [3, -2, 3, -2] | s = [0, 3, 1, 4, 2]
ranks = [0, 1, 2, 3, 4]
R=0: s[R]=0, hạng=1, đã gặp [], query -> 0 (các L = []), tổng = 0
R=1: s[R]=3, hạng=4, đã gặp [0], query -> 1 (các L = [0]), tổng = 1
R=2: s[R]=1, hạng=2, đã gặp [0, 3], query -> 1 (các L = [0]), tổng = 2
R=3: s[R]=4, hạng=5, đã gặp [0, 3, 1], query -> 3 (các L = [0, 1, 2]), tổng = 5
R=4: s[R]=2, hạng=3, đã gặp [0, 3, 1, 4], query -> 2 (các L = [0, 2]), tổng = 7
Đáp án = 7


7

### 8.2. Trace ví dụ 2 và ví dụ 3

Ví dụ 3 (`[2,2,2]`, `a = b = 1`): `arr = [−1,−1,−1]`, `s = [0,−1,−2,−3]` giảm dần nên không có cặp nào `s[L] ≤ s[R]` ⇒ 0.

In [13]:
trace_count_bit(*EXAMPLES[1][:3])
trace_count_bit(*EXAMPLES[2][:3])

arr = [-1, -1, 2] | s = [0, -1, -2, 0]
ranks = [-2, -1, 0]
R=0: s[R]=0, hạng=3, đã gặp [], query -> 0 (các L = []), tổng = 0
R=1: s[R]=-1, hạng=2, đã gặp [0], query -> 0 (các L = []), tổng = 0
R=2: s[R]=-2, hạng=1, đã gặp [0, -1], query -> 0 (các L = []), tổng = 0
R=3: s[R]=0, hạng=3, đã gặp [0, -1, -2], query -> 3 (các L = [0, 1, 2]), tổng = 3
Đáp án = 3
arr = [-1, -1, -1] | s = [0, -1, -2, -3]
ranks = [-3, -2, -1, 0]
R=0: s[R]=0, hạng=4, đã gặp [], query -> 0 (các L = []), tổng = 0
R=1: s[R]=-1, hạng=3, đã gặp [0], query -> 0 (các L = []), tổng = 0
R=2: s[R]=-2, hạng=2, đã gặp [0, -1], query -> 0 (các L = []), tổng = 0
R=3: s[R]=-3, hạng=1, đã gặp [0, -1, -2], query -> 0 (các L = []), tổng = 0
Đáp án = 0


0

### 8.3. Trace merge sort trên `s = [0, 3, 1, 4, 2]`

Chia: `[0,3]`, `[1,4,2]`; ... Hàm dưới in số cặp chéo ở mỗi bước trộn.

In [14]:
def trace_merge(s):
    def rec(arr, depth=0):
        pad = "  " * depth
        if len(arr) <= 1:
            return arr, 0
        mid = len(arr) // 2
        left, cl = rec(arr[:mid], depth + 1)
        right, cr = rec(arr[mid:], depth + 1)
        cross, i = 0, 0
        for v in right:
            while i < len(left) and left[i] <= v:
                i += 1
            cross += i
        merged = sorted(left + right)
        print(f"{pad}gộp trái={left} phải={right}: cặp chéo = {cross} -> {merged}")
        return merged, cl + cr + cross
    res, total = rec(s)
    print("tổng cặp thuận thế =", total)
    return total

trace_merge([0, 3, 1, 4, 2])

  gộp trái=[0] phải=[3]: cặp chéo = 1 -> [0, 3]
    gộp trái=[4] phải=[2]: cặp chéo = 0 -> [2, 4]
  gộp trái=[1] phải=[2, 4]: cặp chéo = 2 -> [1, 2, 4]
gộp trái=[0, 3] phải=[1, 2, 4]: cặp chéo = 4 -> [0, 1, 2, 3, 4]
tổng cặp thuận thế = 7


7

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (BIT + nén toạ độ). Ta viết BIT bằng hai hàm lồng nhau để gọn và nhanh.

In [15]:
from bisect import bisect_left


class Solution:
    def countRatioSubarrays(self, nums: list[int], a: int, b: int) -> int:
        n = len(nums)
        # lẻ -> +a, chẵn -> -b; s là tổng tiền tố
        s = [0] * (n + 1)
        for i, x in enumerate(nums):
            s[i + 1] = s[i] + (a if x % 2 else -b)

        ranks = sorted(set(s))                 # nén toạ độ
        m = len(ranks)
        tree = [0] * (m + 1)

        ans = 0
        for v in s:
            r = bisect_left(ranks, v) + 1      # hạng 1-based của v
            i = r                              # query: số tiền tố đã gặp có hạng <= r
            while i > 0:
                ans += tree[i]
                i -= i & -i
            i = r                              # update: thêm v
            while i <= m:
                tree[i] += 1
                i += i & -i
        return ans


sol = Solution()
for nums, a, b, e in EXAMPLES:
    assert sol.countRatioSubarrays(nums, a, b) == e, (nums, a, b)
print("3 ví dụ của đề: OK")

3 ví dụ của đề: OK


## 10. Vì sao thuật toán đúng

**(a) Biến đổi tương đương.** Với `y ≥ 1` và `b ≥ 1`: `x/y ≤ a/b` ⇔ `x·b ≤ a·y` ⇔ `a·y − b·x ≥ 0` (nhân hai vế với `y·b > 0` giữ chiều bất đẳng thức). Với `y = 0`: đoạn không rỗng nên `x ≥ 1` và `a·y − b·x = −b·x ≤ −1 < 0` ⇒ bất đẳng thức sai, trùng với việc đoạn bị loại bởi `y > 0`. Vậy "hợp lệ" ⇔ `a·y − b·x ≥ 0`.

**(b) Tổng đoạn = tổng giá trị đã thay.** Mỗi lẻ đóng góp `a`, mỗi chẵn đóng góp `−b` nên tổng đoạn đúng bằng `a·y − b·x`.

**(c) Tiền tố.** Mỗi đoạn con khác rỗng ↔ đúng một cặp `(L, R)`, `0 ≤ L < R ≤ n`; tổng đoạn `= s[R] − s[L]`; điều kiện `≥ 0` ⇔ `s[L] ≤ s[R]`.

**(d) BIT đếm đúng.** Trước khi xử lý `R`, BIT chứa các hạng của `s[0..R−1]`; `query(hạng của s[R])` đếm các phần tử có hạng `≤` hạng của `s[R]`, tương đương `s[L] ≤ s[R]` vì nén toạ độ giữ thứ tự (và giá trị bằng nhau có cùng hạng). Cộng cho mọi `R` được tổng số cặp.

**(e) Merge sort đúng.** Mọi cặp `(L, R)` rơi vào đúng một mức đệ quy: cả hai cùng trong một nửa (đếm đệ quy) hoặc chéo (đếm khi trộn). Khi đếm chéo, vì `left` được sắp tăng và `right` cũng sắp tăng, con trỏ `i` chỉ tiến, và `i` bằng số phần tử trái `≤ v`.

**(f) Số nguyên lớn.** `a, b` tới `10^9`, `n` tới `10^5` nên `|s| ≤ 10^14`: số nguyên Python không tràn.

## 11. Phân tích Time/Space

Đặt `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn + `Fraction` | `O(n²)` (hằng số lớn) | `O(1)` |
| 2. Tiền tố, hai vòng lặp | `O(n²)` | `O(n)` |
| 3. BIT + nén toạ độ | `O(n log n)` | `O(n)` |
| 4. Merge sort đếm cặp | `O(n log n)` | `O(n)` |

Ta đo trên `n = 10^5`.

In [16]:
rng = random.Random(4013)
n = 100000
nums = [rng.randint(1, 10**9) for _ in range(n)]
a, b = rng.randint(1, 10**9), rng.randint(1, 10**9)
t = time.time(); r1 = Solution().countRatioSubarrays(nums, a, b); t1 = time.time() - t
t = time.time(); r2 = count_merge_sort(nums, a, b); t2 = time.time() - t
print(f"BIT: {r1} ({t1:.2f}s) | merge sort: {r2} ({t2:.2f}s)")
assert r1 == r2
# Dạng xấu cho BIT: mọi tiền tố tăng (đáp án = n(n+1)/2) và giảm (đáp án 0)
inc = [1] * n; dec = [2] * n
print(Solution().countRatioSubarrays(inc, 1, 1) == n * (n + 1) // 2, Solution().countRatioSubarrays(dec, 1, 1))

BIT: 4999916442 (0.16s) | merge sort: 4999916442 (0.16s)


True 0


## 12. Bộ test

Ví dụ đề, edge case, rồi đối chiếu vét cạn `Fraction` (nguyên văn đề) với tất cả cách trên input ngẫu nhiên có seed. Ta chọn `a, b` nhỏ và `nums` nhỏ để nhiều trường hợp **bằng nhau** (`x/y = a/b` đúng) — kiểm tra dấu `≤`.

In [17]:
ALL = [count_prefix_quadratic, count_bit, count_merge_sort, Solution().countRatioSubarrays]

EDGE = [
    ([1], 1, 1, 1),                  # [1]: x=0, y=1, 0/1 <= 1/1
    ([2], 1, 1, 0),                  # không có số lẻ
    ([1, 2], 1, 1, 2),               # [1] và [1,2] (1/1 <= 1/1: bằng nhau vẫn tính); [2] có y = 0 nên loại
    ([1, 1, 1], 1, 5, 6),            # mọi đoạn có x = 0 -> mọi đoạn hợp lệ
    ([2, 1], 1, 2, 1),               # [2,1]: 1/1 > 1/2 loại; chỉ [1]
    ([2, 1], 2, 1, 2),               # [1] và [2,1] (1/1 <= 2/1)
    ([4, 6, 8, 3], 1, 1, 2),         # [3] (0/1) và [8,3] (1/1 bằng nhau); [6,8,3] có 2/1 > 1
]
for nums, a, b, e in EXAMPLES + EDGE:
    ref = count_bruteforce(nums, a, b)
    assert ref == e, ("trọng tài lệch kỳ vọng", nums, a, b, ref, e)
    for fn in ALL:
        assert fn(nums, a, b) == e, (fn.__name__, nums, a, b, fn(nums, a, b), e)
print("Ví dụ đề + edge case: mọi cách đúng")

rng = random.Random(401301)
equal_cases = 0
for _ in range(1500):
    n = rng.randint(1, 14)
    nums = [rng.randint(1, 6) for _ in range(n)]
    a, b = rng.randint(1, 5), rng.randint(1, 5)
    ref = count_bruteforce(nums, a, b)
    for fn in ALL:
        assert fn(nums, a, b) == ref, (fn.__name__, nums, a, b, fn(nums, a, b), ref)
    # đếm số đoạn có đúng tỉ lệ bằng a/b để chắc dấu "<=" được kiểm
    for i in range(n):
        x = y = 0
        for j in range(i, n):
            if nums[j] % 2 == 0: x += 1
            else: y += 1
            if y > 0 and x * b == a * y:
                equal_cases += 1
print(f"1500 input nhỏ: mọi cách khớp vét cạn (có {equal_cases} đoạn có tỉ lệ đúng bằng a/b -> kiểm tra dấu '<=')")
assert equal_cases > 500

Ví dụ đề + edge case: mọi cách đúng
1500 input nhỏ: mọi cách khớp vét cạn (có 3405 đoạn có tỉ lệ đúng bằng a/b -> kiểm tra dấu '<=')


In [18]:
rng = random.Random(7)
for _ in range(40):
    n = rng.randint(50, 400)
    nums = [rng.randint(1, 10**9) for _ in range(n)]
    a, b = rng.randint(1, 10**9), rng.randint(1, 10**9)
    ref = count_prefix_quadratic(nums, a, b)
    assert ref == Solution().countRatioSubarrays(nums, a, b) == count_merge_sort(nums, a, b)
    if n <= 120:
        assert ref == count_bruteforce(nums, a, b)
print("40 mảng 50..400 phần tử, giá trị lớn: các cách khớp")

nums = [1, 2, 1, 2]
snap = nums[:]
Solution().countRatioSubarrays(nums, 3, 2)
assert nums == snap

40 mảng 50..400 phần tử, giá trị lớn: các cách khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng số thực để so sánh tỉ lệ

**Code sai:** `x / y <= a / b`.
**Vì sao sai:** với `a, b` lớn, `a / b` mất độ chính xác (như mục 3.2), có thể sai khi hai tỉ lệ gần bằng nhau.
**Code đúng:** `x * b <= a * y` (số nguyên) hoặc tổng tiền tố.

In [19]:
def count_float(nums, a, b):
    n = len(nums); total = 0
    for i in range(n):
        x = y = 0
        for j in range(i, n):
            if nums[j] % 2 == 0: x += 1
            else: y += 1
            if y > 0 and x / y <= a / b:          # SAI: số thực
                total += 1
    return total

# a / b gần bằng 1 + 1e-18 nhưng khác 1
a, b = 10**18, 10**18 + 1                          # a/b < 1 một chút
nums = [1, 2]                                      # đoạn [1,2]: x/y = 1/1 = 1 > a/b -> không hợp lệ
print("float:", count_float(nums, a, b), "| chính xác:", count_bruteforce(nums, a, b), "| Solution:", Solution().countRatioSubarrays(nums, a, b))

float: 2 | chính xác: 1 | Solution: 1


### Lỗi 2 — Dùng `<` thay vì `<=` khi đếm cặp tiền tố

**Code sai:** `query(rank - 1)` (chỉ đếm `s[L] < s[R]`).
**Vì sao sai:** tổng đoạn **bằng 0** ứng với `x/y = a/b` đúng, phải tính hợp lệ. Ví dụ 2: `s = [0, −1, −2, 0]` có `s[0] = s[3] = 0`, tức đoạn `[2,2,1]` có `x/y = 2/1 = a/b` — dùng `<` sẽ bỏ sót nó (đáp án 2 thay vì 3).
**Code đúng:** `query(rank)`.

In [20]:
def count_bit_strict(nums, a, b):
    s = [0]
    for x in nums:
        s.append(s[-1] + (a if x % 2 else -b))
    ranks = sorted(set(s))
    bit = Fenwick(len(ranks))
    ans = 0
    for v in s:
        r = bisect_left(ranks, v) + 1
        ans += bit.query(r - 1)                    # SAI: loại các cặp bằng nhau
        bit.update(r, 1)
    return ans

for nums, a, b, e in EXAMPLES:
    print("sai:", count_bit_strict(nums, a, b), "| đúng:", e)

sai: 7 | đúng: 7
sai: 2 | đúng: 3
sai: 0 | đúng: 0


### Lỗi 3 — Quên khởi tạo `s[0] = 0` (không đếm các đoạn bắt đầu từ vị trí 0)

**Code sai:** BIT bắt đầu rỗng và chỉ thêm `s[1..n]`.
**Vì sao sai:** mọi đoạn có `L = 0` bị bỏ.
**Code đúng:** duyệt `s[0..n]` (bao gồm `s[0] = 0`) — BIT thêm `s[0]` ở bước đầu.

### Lỗi 4 — Quên chiều `L < R` (đếm cả cặp `R < L`, hoặc cả `L = R`)

**Code sai:** `update` trước `query` — tính cả `L = R` (đoạn rỗng).
**Code đúng:** `query` **trước**, `update` sau.

In [21]:
def count_bit_update_first(nums, a, b):
    s = [0]
    for x in nums:
        s.append(s[-1] + (a if x % 2 else -b))
    ranks = sorted(set(s))
    bit = Fenwick(len(ranks))
    ans = 0
    for v in s:
        r = bisect_left(ranks, v) + 1
        bit.update(r, 1)                           # SAI: thêm trước khi hỏi -> đếm cả chính nó (L = R)
        ans += bit.query(r)
    return ans

for nums, a, b, e in EXAMPLES:
    print("sai:", count_bit_update_first(nums, a, b), "| đúng:", e, "(thừa n + 1 =", len(nums) + 1, "cặp L = R)")

sai: 12 | đúng: 7 (thừa n + 1 = 5 cặp L = R)
sai: 7 | đúng: 3 (thừa n + 1 = 4 cặp L = R)
sai: 4 | đúng: 0 (thừa n + 1 = 4 cặp L = R)


### Lỗi 5 — Giữ điều kiện `y > 0` riêng khi đổi chiều bất đẳng thức

Xem phần 15 và Bài 5: với `x/y ≥ a/b`, `a·y − b·x ≤ 0` đúng cả với đoạn toàn chẵn (`y = 0`) — phải **trừ** các đoạn toàn chẵn.

## 14. Mẹo & nhận dạng pattern

- **"Tỉ lệ hai đếm so với một hằng số"** → nhân chéo thành tổng tuyến tính `a·y − b·x`, gán trọng số `+a`/`−b` cho từng phần tử.
- **"Đếm đoạn con có tổng ≥ 0 (hoặc ≤ k)"** → tổng tiền tố + đếm cặp thuận thế/nghịch thế (BIT hoặc merge sort).
- **Tổng có thể lớn/âm** → nén toạ độ bằng `sorted(set(...))` + `bisect_left`.
- Luôn xét dấu `≤` hay `<` để chọn `query(rank)` hay `query(rank − 1)`.
- Nhớ kiểm tra điều kiện phụ (`y > 0`) có còn được thoả tự nhiên hay không sau biến đổi.
- Dùng so sánh chính xác (số nguyên / `Fraction`) cho tỉ lệ, không dùng `float`.

## 15. Biến thể

1. **Phiên bản I (`n` nhỏ):** vét cạn `O(n²)` hoặc tiền tố `O(n²)` đủ.
2. **`x / y ≥ a / b` với `y > 0`:** đếm cặp `s[L] ≥ s[R]` rồi **trừ** các đoạn toàn chẵn (Bài 5).
3. **`x / y < a / b` (chặt):** đếm `s[L] < s[R]`, `y > 0` vẫn tự động (đoạn toàn chẵn có tổng `< 0`).
4. **Tỉ lệ `x / (x + y)`** (tỉ lệ số chẵn trên tổng độ dài): `x/(x+y) ≤ a/b` ⇔ `b·x ≤ a·(x + y)` — vẫn tuyến tính, đổi trọng số.
5. **Đếm đoạn con có tổng đúng bằng `0`:** nhóm tiền tố bằng nhau, `Σ C(c, 2)`.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn `Fraction` | So phân số từng đoạn | `O(n²)` | `O(1)` | Trọng tài nguyên văn |
| 2. Tiền tố hai vòng | `s[L] ≤ s[R]` | `O(n²)` | `O(n)` | Kiểm chứng biến đổi |
| 3. BIT + nén toạ độ | Đếm `≤` khi quét `R` | `O(n log n)` | `O(n)` | **Bản nộp** |
| 4. Merge sort | Đếm cặp chéo khi trộn | `O(n log n)` | `O(n)` | Không cần BIT/nén |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `nums = [2,1,1]`, `a = 2`, `b = 1`. Viết mảng `arr` (lẻ → `+a`, chẵn → `−b`) và tổng tiền tố `s`.

**Bài 2 (tính tay).** Với `nums = [2,2,1]`, `a = 2`, `b = 1` (ví dụ 2), liệt kê mọi cặp `(L, R)` có `s[L] ≤ s[R]` và đối chiếu với 3 đoạn hợp lệ.

**Bài 3 (điền TODO).** Hoàn thiện `MyFenwick` (`update`, `query`, 1-based).

**Bài 4 (tự viết lại).** Viết `my_count(nums, a, b)` bằng BIT + nén toạ độ.

**Bài 5 (biến thể).** Đếm số đoạn con có `y > 0` và `x / y ≥ a / b`. Viết `count_ge(nums, a, b)`. Gợi ý: `a·y − b·x ≤ 0`; đoạn toàn chẵn cũng thoả `≤ 0` nhưng `y = 0` ⇒ phải trừ.

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
class MyFenwick:
    def __init__(self, n):
        # TODO: lưu n, tạo mảng t độ dài n + 1 toàn 0
        pass
    def update(self, i, delta):
        # TODO: while i <= n: t[i] += delta; i += i & -i
        pass
    def query(self, i):
        # TODO: while i > 0: s += t[i]; i -= i & -i
        pass

# Bài 4 — TODO
def my_count(nums, a, b):
    # TODO: tiền tố s, nén toạ độ, duyệt s: ans += query(rank) rồi update(rank, 1)
    pass

# Bài 5 — TODO
def count_ge(nums, a, b):
    # TODO: đếm cặp s[L] >= s[R] (L < R), rồi trừ số đoạn toàn chẵn
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `2` chẵn → `−1`; `1` lẻ → `+2`.
- Mức 2: `arr = [−1, 2, 2]`.
- Mức 3: `s = [0, −1, 1, 3]`.

**Bài 2.**
- Mức 1: `arr = [−1, −1, 2]`, `s = [0, −1, −2, 0]`.
- Mức 2: chỉ `R = 3` (`s = 0`) có `L` thoả.
- Mức 3: các cặp `(0,3)`, `(1,3)`, `(2,3)` tương ứng đoạn `[2,2,1]`, `[2,1]`, `[1]` — đúng 3 đoạn của ví dụ 2.

**Bài 3.**
- Mức 1: xem lớp `Fenwick` ở mục 3.6.
- Mức 2: `update` đi lên (`+= i & -i`), `query` đi xuống (`-= i & -i`).
- Mức 3: hạng 1-based.

**Bài 4.**
- Mức 1: tiền tố `s` gồm `n + 1` giá trị.
- Mức 2: `ranks = sorted(set(s))`, `bisect_left(ranks, v) + 1`.
- Mức 3: `query` **trước** rồi `update`.

**Bài 5.**
- Mức 1: đổi chiều thành `s[L] ≥ s[R]`: đếm "bằng hoặc lớn hơn".
- Mức 2: số `s[L] ≥ v` đã gặp = `đã gặp − query(rank − 1)`.
- Mức 3: số đoạn toàn chẵn trong một đoạn run độ dài `k` là `k(k + 1)/2`; trừ tổng các run.

## 19. Đáp án tham khảo

In [23]:
# Bài 1, 2
arr = [(2 if x % 2 else -1) for x in [2, 1, 1]]
assert arr == [-1, 2, 2] and prefix_sums(arr) == [0, -1, 1, 3]
s = prefix_sums([-1, -1, 2])
pairs = [(L, R) for R in range(len(s)) for L in range(R) if s[L] <= s[R]]
assert pairs == [(0, 3), (1, 3), (2, 3)]
assert Solution().countRatioSubarrays([2, 2, 1], 2, 1) == 3

# Bài 3
class MyFenwickAns:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)
    def update(self, i, delta):
        while i <= self.n:
            self.t[i] += delta
            i += i & -i
    def query(self, i):
        s = 0
        while i > 0:
            s += self.t[i]
            i -= i & -i
        return s

f1, f2 = Fenwick(30), MyFenwickAns(30)
rr = random.Random(5)
for _ in range(300):
    p, d = rr.randint(1, 30), rr.randint(-3, 3)
    f1.update(p, d); f2.update(p, d)
    q_ = rr.randint(0, 30)
    assert f1.query(q_) == f2.query(q_)
print("Bài 1–3 OK")

Bài 1–3 OK


In [24]:
# Bài 4
def my_count_ans(nums, a, b):
    s = [0]
    for x in nums:
        s.append(s[-1] + (a if x % 2 else -b))
    ranks = sorted(set(s))
    bit = MyFenwickAns(len(ranks))
    ans = 0
    for v in s:
        r = bisect_left(ranks, v) + 1
        ans += bit.query(r)
        bit.update(r, 1)
    return ans

# Bài 5
def count_ge_ans(nums, a, b):
    s = [0]
    for x in nums:
        s.append(s[-1] + (a if x % 2 else -b))
    ranks = sorted(set(s))
    bit = MyFenwickAns(len(ranks))
    pairs_ge = 0                                    # số cặp L < R có s[L] >= s[R]
    seen = 0
    for v in s:
        r = bisect_left(ranks, v) + 1
        pairs_ge += seen - bit.query(r - 1)         # đã gặp trừ số có s[L] < v
        bit.update(r, 1)
        seen += 1
    all_even = 0                                    # đoạn toàn chẵn: tổng k(k+1)/2 theo từng dãy chẵn liên tiếp
    run = 0
    for x in nums:
        if x % 2 == 0:
            run += 1
            all_even += run
        else:
            run = 0
    return pairs_ge - all_even

def count_ge_bruteforce(nums, a, b):
    n, total, ratio = len(nums), 0, Fraction(a, b)
    for i in range(n):
        x = y = 0
        for j in range(i, n):
            if nums[j] % 2 == 0: x += 1
            else: y += 1
            if y > 0 and Fraction(x, y) >= ratio:
                total += 1
    return total

rng = random.Random(99)
for _ in range(400):
    n = rng.randint(1, 12)
    nums = [rng.randint(1, 6) for _ in range(n)]
    a, b = rng.randint(1, 5), rng.randint(1, 5)
    assert my_count_ans(nums, a, b) == Solution().countRatioSubarrays(nums, a, b)
    assert count_ge_ans(nums, a, b) == count_ge_bruteforce(nums, a, b), (nums, a, b)
print("Bài 4–5 OK (bài 5 đối chiếu vét cạn Fraction)")

Bài 4–5 OK (bài 5 đối chiếu vét cạn Fraction)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
# x/y <= a/b, y > 0   <=>   a*y - b*x >= 0
s[0] = 0;  s[i+1] = s[i] + (a if nums[i] % 2 else -b)
ans = #{(L, R): L < R, s[L] <= s[R]}

ranks = sorted(set(s))
for v in s:                      # R = 0..n
    r = bisect_left(ranks, v) + 1
    ans += bit.query(r)          # hỏi TRƯỚC
    bit.update(r, 1)             # thêm SAU
```

**Checklist trước khi submit:**

- [ ] Biến đổi `x/y ≤ a/b` ⇔ `a·y − b·x ≥ 0`; kiểm tra `y > 0` đã được bao gồm.
- [ ] Lẻ → `+a`, chẵn → `−b` (không đảo).
- [ ] `s[0] = 0` có mặt; duyệt cả `s[0..n]`.
- [ ] Dấu `≤`: `query(rank)`; `query` trước `update`.
- [ ] Nén toạ độ (`sorted(set)` + `bisect_left`) vì giá trị tới `10^14`.
- [ ] Không dùng `float` cho tỉ lệ.
- [ ] Đối chiếu vét cạn `Fraction` trên input nhỏ, nhiều trường hợp tỉ lệ bằng nhau.